# Laboratorium 2: Współbieżność i Równoległość w Pythonie

### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---


## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---


### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).**

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`.

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:

1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).


In [3]:
import requests
from bs4 import BeautifulSoup
import time


def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip()
                    for item in soup.select('.item__link h3')]
    return event_titles


def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]

    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()

    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))

    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")

    print(f"\nCzas wykonania: {time.time() - start:.2f}s")


run_sequential_demo()

c:\Users\tomcz\miniconda3\envs\architektura\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Frogshop
2. Mars
3. Wszystko, co najlepsze
4. Królowa
5. Krako(w)skie Klasyki
6. KIERMASH | Wiosna na Kazimierzu
7. Akira Kurosawa: Droga przez czas. Przegląd filmów w Kinie Kika
8. Wielki Gatsby
9. Dwa Sławy: Będzie dobrze w Klubie Spotkań Poczta Główna
10. Martwe natury

Czas wykonania: 6.15s


In [4]:
import concurrent.futures


def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]

    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))

    all_titles = [title for sublist in results for title in sublist]

    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")

    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")


run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Folwark zwierzęcy
2. Wszystko, co najlepsze
3. Królowa
4. XV Festiwal Filmu Filozoficznego: Współczesna duchowość
5. Kompromitacja, SKAcowani, ADHD Syndrom, SKA Petarda: IV rozbiór Małopolski
6. Targi Japońskie 2026
7. Wielki Gatsby
8. Krako(w)skie Klasyki
9. Moja miłość
10. Martwe natury

Czas wykonania (wątki): 0.98s


---

## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).


In [5]:
import threading


class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001)  # Symulacja opóźnienia
            self.balance = current + amount


account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))

print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


---

## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.


In [6]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes


def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()

    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)

    print(f"Zakończono w czasie {time.time() - start:.2f}s.")


if __name__ == "__main__":
    run_primes_demo()

Praca na 18 procesach (rdzeniach)...
Zakończono w czasie 0.90s.


---

# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.


### Zadanie 1 (Threading)

Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:

1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

_Podpowiedź: Użyj `requests.get(URL).json().get('fact')`_


In [1]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures
import functools

CAT_API_URL = "https://catfact.ninja/fact"
TIMEOUT = 5
FACTS_COUNT = 20


def get_cat_fact():
    return requests.get(CAT_API_URL, timeout=5).json().get('fact')


def handle_request(func):
    functools.wraps(func)

    def wrapper(*args, **kwargs):
        try:
            start = time.time()
            func()

        except requests.HTTPError as http_err:
            print(f'HTTP error: {http_err}')
        except requests.ConnectionError as conn_err:
            print(f'Connection error: {conn_err}')
        except requests.Timeout as timeout_err:
            print(f'Request timeout after {TIMEOUT}s : {timeout_err}')
        except Exception as err:
            print(f'Unknown error: {err}')
        finally:
            print(f'\nCzas wykonania: {time.time() - start:.2f}s')
    return wrapper


@handle_request
def cat_facts_sequential():
    for i in range(1, FACTS_COUNT + 1):
        response = get_cat_fact()
        print(f'{i}. {response}')


# cat_facts_sequential()


@handle_request
def cat_facts_threded():
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(get_cat_fact) for _ in range(FACTS_COUNT)]
        results = [f.result() for f in futures]  # zachowuje kolejność
        # results = [f.result() for f in concurrent.futures.as_completed(futures)] # nie zachowuje kolejności ale zwraca wyniki na bieżąco

    for i, fact in enumerate(results, 1):
        print(f'{i}. {fact}')


cat_facts_threded()

c:\Users\tomcz\miniconda3\envs\architektura\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


1. It has been scientifically proven that owning cats is good for our health and can decrease the occurrence of high blood pressure and other illnesses.
2. When a cat drinks, its tongue - which has tiny barbs on it - scoops the liquid up backwards.
3. The first official cat show in the UK was organised at Crystal Palace in 1871.
4. Unlike dogs, cats do not have a sweet tooth. Scientists believe this is due to a mutation in a key taste receptor.
5. A cat's brain is more similar to a man's brain than that of a dog.
6. Cats only use their meows to talk to humans, not each other. The only time they meow to communicate with other felines is when they are kittens to signal to their mother.
7. When your cats rubs up against you, she is actually marking you as 'hers' with her scent. If your cat pushes his face against your head, it is a sign of acceptance and affection.
8. Cats can jump up to 7 times their tail length.
9. A female cat will be pregnant for approximately 9 weeks or between 62 an

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)

Napisz program o strukturze **producent-consumers**:

1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.


In [6]:
from queue import Queue
import threading
import time

MAX_ELEMENTS = 10


def producer(odd_queue: Queue, even_queue: Queue):
    for i in range(MAX_ELEMENTS):
        if i % 2 == 0:
            even_queue.put(i)
        else:
            odd_queue.put(i)
        print(f'{i} produced and put to a queue')
        time.sleep(0.2)


def odd_consumer(odd_queue: Queue):
    while True:
        next_element = odd_queue.get()
        print(f'{next_element} consumed by odd_consumer')
        odd_queue.task_done()


def even_consumer(even_queue: Queue):
    while True:
        next_element = even_queue.get()
        print(f'{next_element} consumed by even_consumer')
        even_queue.task_done()


if __name__ == '__main__':
    odd_queue = Queue()
    even_queue = Queue()
    t_prod = threading.Thread(target=producer, args=(
        odd_queue, even_queue))
    t_prod.start()

    threading.Thread(target=odd_consumer, args=(
        odd_queue,), daemon=True).start()
    threading.Thread(target=even_consumer, args=(
        even_queue,), daemon=True).start()

    t_prod.join()
    odd_queue.join()
    even_queue.join()

0 produced and put to a queue
0 consumed by even_consumer
1 produced and put to a queue
1 consumed by odd_consumer
2 produced and put to a queue2 consumed by even_consumer

3 produced and put to a queue
3 consumed by odd_consumer
4 produced and put to a queue4 consumed by even_consumer

5 produced and put to a queue
5 consumed by odd_consumer
6 produced and put to a queue6 consumed by even_consumer

7 produced and put to a queue
7 consumed by odd_consumer
8 produced and put to a queue8 consumed by even_consumer

9 produced and put to a queue
9 consumed by odd_consumer


### Zadanie 3 (Multiprocessing)

Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).


In [10]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

CALC_RANGE = range(1, 3)

if __name__ == "__main__":
    with multiprocessing.Pool(processes=3) as pool:
        results = pool.map(calculate_power_sum, CALC_RANGE)

    print(results)

[100, 2535301200456458802993406410750]
